# No Language Left Behind: A Deep Dive into Scaling Machine Translation

### An Interactive Educational Notebook

This notebook explores the core concepts from the research paper **"No Language Left Behind: Scaling Human-Centered Machine Translation"** by the NLLB Team at Meta AI. We will deconstruct the key innovations that enabled the creation of a machine translation model covering over 200 languages, with a special focus on low-resource languages.

## Section 1: Overview & Prerequisites

### 1.1 Summary of the Research

The "No Language Left Behind" (NLLB) project tackled the immense challenge of creating a single, high-quality machine translation (MT) system for over 200 languages, many of which are "low-resource" (meaning they lack large-scale digital text data). The work represents a significant leap from previous systems that covered around 100 languages.

The core contributions can be broken down into three main areas:

1.  **Data Curation & Creation:** Recognizing that data is the primary bottleneck, the project developed novel techniques to create vast training and evaluation datasets. This included:
    *   **FLORES-200:** A high-quality, human-translated evaluation benchmark covering all target languages, enabling reliable progress measurement.
    *   **Large-Scale Bitext Mining:** A sophisticated pipeline to find parallel sentences (translations) from massive, noisy web data (Common Crawl). This pipeline relies on highly accurate Language Identification (LID) and advanced sentence encoders (LASER3) built using a teacher-student distillation approach.
    *   **Backtranslation:** Using strong initial models to generate synthetic training data for low-resource languages, further boosting their data availability.

2.  **Modeling Innovations:** To handle the scale and diversity of 200+ languages, the project introduced advanced modeling techniques:
    *   **Mixture-of-Experts (MoE) Models:** Using a 54.5 billion parameter Sparsely Gated MoE Transformer to massively increase model capacity without a proportional increase in computational cost, allowing experts to specialize in different languages or language families.
    *   **Advanced Regularization & Curriculum Learning:** Developing novel methods like MoE Expert Output Masking (EOM) and phased training curricula to combat the severe overfitting that large models experience on low-resource languages.

3.  **Holistic & Human-Centered Evaluation:** Going beyond standard metrics like BLEU, the project focused on:
    *   **Standardized Human Evaluation:** Using the XSTS protocol with calibration sets to ensure human judgments of quality were consistent and comparable across dozens of language pairs.
    *   **Toxicity Detection:** Creating comprehensive toxicity wordlists for all 200 languages to measure and mitigate the risk of models adding offensive content during translation.
    
This notebook will walk through the mathematical foundations, prerequisite algorithms, and core implementations of these key contributions.

### 1.2 Prerequisites

To fully grasp the concepts in this notebook, the following background knowledge is recommended.

**Mathematical Concepts:**
- **Linear Algebra:** Vector spaces, dot products, matrix multiplication, norms.
- **Calculus:** Partial derivatives, gradients, the chain rule (for backpropagation).
- **Probability & Statistics:** Probability distributions, conditional probability, expectation.
- **Information Theory:** Entropy, cross-entropy, perplexity.

**Machine Learning & Computer Science Concepts:**
- **Neural Networks:** Basic architecture (layers, neurons, activation functions), loss functions, gradient descent.
- **Natural Language Processing (NLP):** Tokenization (subword units), embeddings, sequence-to-sequence models.
- **The Transformer Architecture:** Self-attention mechanism, multi-head attention, positional encodings, encoder-decoder structure.
- **Knowledge Distillation:** The concept of a larger "teacher" model training a smaller "student" model.
- **Mixture of Experts (MoE):** Basic understanding of conditional computation where only parts of a network are activated per input.

### 1.3 Notebook Structure & Learning Objectives

**Hierarchy of Topics:**
1.  **Overview & Prerequisites**: Setting the stage for our deep dive.
2.  **Mathematical Foundations**: Implementing key math concepts like Cosine Similarity and Cross-Entropy from scratch.
3.  **Prerequisite Algorithms**: Building a foundational Transformer model from scratch to understand the base architecture.
4.  **Core Research Content**: Implementing the key NLLB innovations:
    - The Bitext Mining pipeline philosophy (LID, Sentence Encoders).
    - Sparsely Gated Mixture-of-Experts (MoE) layers.
    - Advanced regularization with Expert Output Masking (EOM).
5.  **Experimental Analysis**: Visually reproducing key results from the paper, such as the overfitting problem in MoE models and the effect of different data sources.
6.  **Research Context & Extensions**: Exploring model distillation for creating smaller, practical models and discussing the challenges of dialectal translation.

**Learning Objectives:**
- **Understand** the core challenges of low-resource machine translation.
- **Implement** a Transformer model and a Mixture-of-Experts layer.
- **Grasp** the importance of data mining and augmentation for large-scale multilingual models.
- **Analyze** the trade-offs between model scale, performance, and overfitting.
- **Appreciate** the need for human-centered and safety-focused evaluation.

**Estimated Time:** 2-3 hours.

## Section 2: Mathematical Foundations

Before diving into complex models, let's implement and visualize the core mathematical tools used in the NLLB paper, particularly for the bitext mining and model training processes.

### 2.1 Cosine Similarity: Measuring Meaning

The entire bitext mining pipeline hinges on finding sentences with similar meanings across languages. This is achieved by representing sentences as high-dimensional vectors (embeddings) and measuring their similarity. The most common metric for this is **Cosine Similarity**, which measures the cosine of the angle between two vectors. A value of 1 means the vectors point in the same direction (identical meaning), 0 means they are orthogonal (unrelated), and -1 means they are opposite.

The formula is: 
$$ \text{similarity} = \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|} = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \sqrt{\sum_{i=1}^{n} B_i^2}} $$

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def educational_cosine_similarity(vec_a, vec_b):
    """
    Clear, step-by-step implementation of cosine similarity for understanding.
    - Based directly on the mathematical formula.
    - Uses basic loops and operations.
    """
    dot_product = 0.0
    norm_a = 0.0
    norm_b = 0.0
    
    for i in range(len(vec_a)):
        dot_product += vec_a[i] * vec_b[i]
        norm_a += vec_a[i]**2
        norm_b += vec_b[i]**2
        
    norm_a = np.sqrt(norm_a)
    norm_b = np.sqrt(norm_b)
    
    if norm_a == 0 or norm_b == 0:
        return 0.0 # Avoid division by zero
        
    return dot_product / (norm_a * norm_b)

def optimized_cosine_similarity(vec_a, vec_b):
    """
    Efficient, vectorized implementation using NumPy.
    """
    # Ensure inputs are NumPy arrays
    vec_a = np.asarray(vec_a)
    vec_b = np.asarray(vec_b)
    
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    
    if norm_a == 0 or norm_b == 0:
        return 0.0
        
    return dot_product / (norm_a * norm_b)

# --- Interactive Visualization ---
@widgets.interact(angle=widgets.FloatSlider(value=45, min=0, max=180, step=1, description='Angle (θ):'))
def interactive_cosine_explorer(angle):
    """
    Interactive widget to visualize cosine similarity between two 2D vectors.
    """
    # Vector A is fixed along the x-axis
    vec_a = np.array([1, 0])
    
    # Vector B is rotated based on the angle slider
    angle_rad = np.deg2rad(angle)
    vec_b = np.array([np.cos(angle_rad), np.sin(angle_rad)])
    
    # Calculate similarity
    sim_educational = educational_cosine_similarity(vec_a, vec_b)
    sim_optimized = optimized_cosine_similarity(vec_a, vec_b)
    
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.arrow(0, 0, vec_a[0], vec_a[1], head_width=0.05, head_length=0.1, fc='blue', ec='blue', label='Vector A')
    ax.arrow(0, 0, vec_b[0], vec_b[1], head_width=0.05, head_length=0.1, fc='red', ec='red', label='Vector B')
    
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-0.2, 1.2)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True)
    ax.axhline(0, color='black',linewidth=0.5)
    ax.axvline(0, color='black',linewidth=0.5)
    ax.set_title(f'Cosine Similarity: {sim_optimized:.2f}')
    ax.legend()
    plt.show()

    print(f"Educational implementation result: {sim_educational:.4f}")
    print(f"Optimized implementation result:   {sim_optimized:.4f}")

### 2.2 Cross-Entropy Loss: Guiding the Model's Learning

The machine translation model learns by trying to predict the next word in the translated sentence. The **Cross-Entropy Loss** function measures how well the probability distribution predicted by the model matches the actual target distribution (which is a one-hot vector, where the correct word has a probability of 1 and all others have 0).

For a single prediction, the formula is:
$$ H(p, q) = - \sum_{i=1}^{C} p(x_i) \log q(x_i) $$ 

Where:
- $ C $ is the number of classes (the vocabulary size).
- $ p(x_i) $ is the true probability of class $i$ (1 for the correct word, 0 for all others).
- $ q(x_i) $ is the model's predicted probability for class $i$ (output of a Softmax function).

This simplifies to just the negative log probability of the correct word: $ -\log(q(x_{\text{correct}})) $. Minimizing this loss forces the model to assign a higher probability to the correct word.

In [ ]:
def educational_softmax(logits):
    """Clear implementation of the softmax function."""
    exps = [np.exp(logit) for logit in logits]
    sum_of_exps = sum(exps)
    softmax_probs = [exp / sum_of_exps for exp in exps]
    return softmax_probs

def educational_cross_entropy_loss(predicted_probs, target_index):
    """Clear implementation of cross-entropy loss for a single example."""
    # The loss is simply the negative log of the probability of the correct class
    correct_prob = predicted_probs[target_index]
    return -np.log(correct_prob)

def optimized_cross_entropy_loss(logits_tensor, target_index_tensor):
    """Efficient implementation using PyTorch's built-in functions."""
    # PyTorch's F.cross_entropy handily combines log_softmax and NLLLoss
    return F.cross_entropy(logits_tensor, target_index_tensor)

# --- Example Usage and Verification ---
# Model's raw output for a vocabulary of 5 words
logits = [2.0, 1.0, 0.1, -1.0, 0.5]
target_index = 0 # The correct word is the first one

# Educational Path
probs_edu = educational_softmax(logits)
loss_edu = educational_cross_entropy_loss(probs_edu, target_index)

# Optimized Path (using PyTorch)
logits_pt = torch.tensor([logits], dtype=torch.float32) # Add batch dimension
target_pt = torch.tensor([target_index], dtype=torch.long)
loss_pt = optimized_cross_entropy_loss(logits_pt, target_pt).item()

print(f"Logits: {logits}")
print(f"Target Index: {target_index}\n")
print(f"Probabilities (Educational): {[f'{p:.3f}' for p in probs_edu]}\n")
print(f"Educational Cross-Entropy Loss: {loss_edu:.4f}")
print(f"Optimized Cross-Entropy Loss:   {loss_pt:.4f}")

## Section 3: Prerequisite Algorithms

The NLLB-200 model is a variant of the Transformer. To understand its architecture and the innovations applied to it, we must first understand the original "Attention Is All You Need" Transformer model.

### 3.1 The Transformer Architecture (From Scratch)

The Transformer, introduced by Vaswani et al. (2017), revolutionized sequence-to-sequence tasks. Its core innovation is the **self-attention mechanism**, which allows the model to weigh the importance of different words in the input sequence when processing a specific word. This replaces the recurrent connections of models like LSTMs, enabling massive parallelization.

We will implement the key components from scratch to build a functional (though not optimized) Transformer block.

**Key Components:**
1.  **Scaled Dot-Product Attention:** The core of the attention mechanism. It computes attention scores for a set of queries ($Q$), keys ($K$), and values ($V$).
    $$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$ 
2.  **Multi-Head Attention:** Runs the attention mechanism in parallel multiple times (with different linear projections) and concatenates the results, allowing the model to focus on different aspects of the sequence.
3.  **Position-wise Feed-Forward Network:** A simple two-layer fully connected network applied independently to each position.
4.  **Positional Encoding:** Since the model contains no recurrence, we add information about the position of tokens in the sequence to the input embeddings.

In [ ]:
import torch
import torch.nn as nn
import math

class EducationalMultiHeadAttention(nn.Module):
    """
    Educational implementation of Multi-Head Attention.
    Focuses on clarity over performance.
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear layers for Query, Key, Value, and the final output
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """The core attention mechanism."""
        # 1. MatMul Q and K.T
        attn_scores = torch.matmul(Q, K.transpose(-2, -1))
        
        # 2. Scale by sqrt(d_k)
        attn_scores = attn_scores / math.sqrt(self.d_k)
        
        # 3. Apply mask (optional, for decoder)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        # 4. Apply softmax
        attn_probs = torch.softmax(attn_scores, dim=-1)
        
        # 5. MatMul with V
        output = torch.matmul(attn_probs, V)
        return output, attn_probs

    def forward(self, x):
        batch_size, seq_length, _ = x.size()
        
        # 1. Pass input through linear layers
        Q = self.w_q(x)
        K = self.w_k(x)
        V = self.w_v(x)
        
        # 2. Reshape for multi-head attention
        # (batch_size, seq_length, d_model) -> (batch_size, num_heads, seq_length, d_k)
        Q = Q.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. Apply scaled dot-product attention
        attn_output, attn_probs = self.scaled_dot_product_attention(Q, K, V)
        
        # 4. Concatenate heads and pass through final linear layer
        # (batch_size, num_heads, seq_length, d_k) -> (batch_size, seq_length, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        output = self.w_o(attn_output)
        
        return output, attn_probs

# --- Let's test it ---
d_model = 512
num_heads = 8
batch_size = 2
seq_len = 10

attention = EducationalMultiHeadAttention(d_model, num_heads)
input_tensor = torch.randn(batch_size, seq_len, d_model)
output, attn_probs = attention(input_tensor)

print(f"Input shape: {input_tensor.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention probabilities shape (one head): {attn_probs.shape}")

# Visualize attention for the first head of the first batch item
plt.figure(figsize=(8, 6))
plt.imshow(attn_probs[0, 0].detach().numpy(), cmap='viridis')
plt.xlabel("Key Position")
plt.ylabel("Query Position")
plt.title("Self-Attention Weights (Head 1, Batch 1)")
plt.colorbar()
plt.show()

## Section 4: Core Research Content

Now we will implement simplified, educational versions of the core modeling innovations from the NLLB paper: Sparsely Gated Mixture-of-Experts, the EOM regularization technique, and the concept of curriculum learning.

### 4.1 Sparsely-Gated Mixture-of-Experts (MoE) Layer

A standard Transformer uses the same Feed-Forward Network (FFN) for every token. An MoE model replaces this FFN with a set of parallel FFNs, called "experts". For each token, a small trainable "gating network" decides which expert(s) to send it to. The NLLB model uses a **Top-2 Gating** strategy, where each token is processed by the two experts that the gating network scores most highly.

The output is a weighted sum of the outputs from the selected experts:
$$ \text{MoE}(x_t) = \sum_{e=1}^{E} G(x_t)_e \cdot \text{FFN}_e(x_t) $$

Where $G(x_t)_e$ is the gate's weight for expert $e$. In Top-2 gating, only the top two weights are non-zero.

A crucial component is the **Load Balancing Loss**, an auxiliary loss that encourages the gating network to send a roughly equal number of tokens to each expert, preventing a scenario where only a few experts are ever used.

In [ ]:
class EducationalExpert(nn.Module):
    """A simple Feed-Forward Network, representing one expert."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.activation = nn.ReLU()
        
    def forward(self, x):
        return self.fc2(self.activation(self.fc1(x)))

class EducationalMoELayer(nn.Module):
    """
    Educational implementation of a Top-2 Sparsely Gated MoE layer.
    """
    def __init__(self, d_model, num_experts, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        
        # The list of expert networks
        self.experts = nn.ModuleList([EducationalExpert(d_model, d_model * 4) for _ in range(num_experts)])
        
        # The gating network is a simple linear layer
        self.gate = nn.Linear(d_model, num_experts)
        
    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        batch_size, seq_len, d_model = x.shape
        x_flat = x.view(-1, d_model) # Reshape to (num_tokens, d_model)
        num_tokens = x_flat.shape[0]

        # 1. Get gating scores for each token
        gate_logits = self.gate(x_flat) # (num_tokens, num_experts)
        gate_probs = F.softmax(gate_logits, dim=1)
        
        # 2. Select Top-K experts for each token
        top_k_probs, top_k_indices = torch.topk(gate_probs, self.top_k, dim=1)
        
        # 3. Normalize the weights of the top-k experts
        top_k_probs = top_k_probs / torch.sum(top_k_probs, dim=1, keepdim=True)
        
        # 4. Route tokens to experts and combine outputs
        final_output = torch.zeros_like(x_flat)
        for i in range(num_tokens):
            token_input = x_flat[i]
            token_output = torch.zeros(d_model)
            for k in range(self.top_k):
                expert_idx = top_k_indices[i, k].item()
                expert_weight = top_k_probs[i, k]
                
                # Get output from the selected expert
                expert_output = self.experts[expert_idx](token_input)
                
                # Weight the expert's output
                token_output += expert_weight * expert_output
            final_output[i] = token_output
            
        # --- Load Balancing Loss Calculation (simplified) ---
        # This encourages gates to use all experts equally.
        # The paper uses a more complex formula, but this captures the idea.
        tokens_per_expert = F.one_hot(top_k_indices[:, 0], num_classes=self.num_experts).float().sum(0)
        load_balancing_loss = torch.std(tokens_per_expert) / torch.mean(tokens_per_expert)
        
        return final_output.view(batch_size, seq_len, d_model), load_balancing_loss, gate_probs

# --- Let's test and visualize the gating ---
d_model = 64
num_experts = 8
moe_layer = EducationalMoELayer(d_model, num_experts)
input_tensor = torch.randn(1, 10, d_model) # 1 sentence of 10 tokens
output, loss, gate_probs = moe_layer(input_tensor)

print(f"Input shape: {input_tensor.shape}")
print(f"Output shape: {output.shape}")
print(f"Load Balancing Loss: {loss.item():.4f}\n")

plt.figure(figsize=(10, 4))
plt.imshow(gate_probs.detach().numpy().T, cmap='viridis', aspect='auto')
plt.xlabel("Token Position in Sequence")
plt.ylabel("Expert ID")
plt.title("Gating Probabilities per Token")
plt.colorbar(label="Probability")
plt.show()

### 4.2 Expert Output Masking (EOM) for Regularization

As the paper notes, large MoE models are prone to overfitting on low-resource languages. Standard dropout is not always sufficient. The paper proposes **MoE Expert Output Masking (EOM)**, a specialized regularization technique.

The idea is simple: for a random fraction of tokens, the outputs of the selected experts are masked (zeroed out) *before* they are combined. This forces the model to rely more on its residual connection (the skip-connection around the MoE block) and reduces co-adaptation between experts.

Let's modify our MoE layer to include this technique.

In [ ]:
class EducationalMoELayerWithEOM(EducationalMoELayer):
    """
    Extends the MoE layer to include Expert Output Masking (EOM).
    """
    def __init__(self, d_model, num_experts, top_k=2, eom_prob=0.1):
        super().__init__(d_model, num_experts, top_k)
        self.eom_prob = eom_prob

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape
        x_flat = x.view(-1, d_model)
        num_tokens = x_flat.shape[0]

        gate_logits = self.gate(x_flat)
        gate_probs = F.softmax(gate_logits, dim=1)
        top_k_probs, top_k_indices = torch.topk(gate_probs, self.top_k, dim=1)
        top_k_probs = top_k_probs / torch.sum(top_k_probs, dim=1, keepdim=True)

        final_output = torch.zeros_like(x_flat)
        for i in range(num_tokens):
            token_input = x_flat[i]
            token_output = torch.zeros(d_model)
            
            # --- EOM Logic ---
            # Decide if we should mask the expert outputs for this token
            apply_eom = torch.rand(1) < self.eom_prob
            
            for k in range(self.top_k):
                expert_idx = top_k_indices[i, k].item()
                expert_weight = top_k_probs[i, k]
                expert_output = self.experts[expert_idx](token_input)
                
                if not apply_eom:
                    token_output += expert_weight * expert_output
                # If apply_eom is True, we add nothing, effectively masking the output.
                
            final_output[i] = token_output
        
        # Load balancing loss is calculated the same way, as it depends on gate scores, not outputs
        tokens_per_expert = F.one_hot(top_k_indices[:, 0], num_classes=self.num_experts).float().sum(0)
        load_balancing_loss = torch.std(tokens_per_expert) / torch.mean(tokens_per_expert)

        return final_output.view(batch_size, seq_len, d_model), load_balancing_loss

# --- Demonstration ---
eom_prob = 0.5 # Set high for clear demonstration
moe_eom_layer = EducationalMoELayerWithEOM(d_model, num_experts, eom_prob=eom_prob)
output_eom, _ = moe_eom_layer(input_tensor)

print("EOM forces some token outputs to be zero (relying on the residual connection).")
print("Original Model Output (first 5 tokens):\n", output[0, :5, :4].detach().numpy().round(2))
print("\nModel with EOM Output (first 5 tokens):\n", output_eom[0, :5, :4].detach().numpy().round(2))

## Section 5: Experimental Analysis

A key finding of the paper is that while MoE models offer massive capacity, they overfit easily on low-resource language pairs. Let's create a simplified experiment to visualize this phenomenon, reproducing the core insight of Figure 17 from the paper.

### 5.1 Visualizing Overfitting in Low-Resource MoE Training

We'll set up a toy translation task with two language pairs:
1.  **High-Resource (`eng` -> `fra`):** A larger dataset.
2.  **Low-Resource (`eng` -> `wol`):** A much smaller dataset.

We will train three small models on this combined data:
1.  A **Dense** Transformer.
2.  An **MoE** Transformer.
3.  An **MoE** Transformer with **EOM** regularization.

We will then plot the validation loss (a proxy for perplexity) for both the high-resource and low-resource pairs over the course of training. We expect to see the MoE model's loss on the low-resource pair start to increase after an initial decrease, indicating overfitting. The EOM and Dense models should show more stable performance.

In [ ]:
import random

# --- Toy Data Generation ---
def generate_toy_data(src_vocab_size, tgt_vocab_size, num_samples, seq_len):
    # Simple task: learn to reverse a sequence
    src_data = torch.randint(1, src_vocab_size, (num_samples, seq_len))
    tgt_data = torch.flip(src_data, [1])
    return src_data, tgt_data

VOCAB_SIZE = 100
SEQ_LEN = 8
D_MODEL = 32
NUM_HEADS = 4
NUM_EXPERTS = 4

# High-resource: 2000 samples
hr_src, hr_tgt = generate_toy_data(VOCAB_SIZE, VOCAB_SIZE, 2000, SEQ_LEN)
# Low-resource: 200 samples
lr_src, lr_tgt = generate_toy_data(VOCAB_SIZE, VOCAB_SIZE, 200, SEQ_LEN)

# Create validation sets
hr_val_src, hr_val_tgt = generate_toy_data(VOCAB_SIZE, VOCAB_SIZE, 100, SEQ_LEN)
lr_val_src, lr_val_tgt = generate_toy_data(VOCAB_SIZE, VOCAB_SIZE, 100, SEQ_LEN)

# Combine data for training
all_src = torch.cat([hr_src, lr_src], dim=0)
all_tgt = torch.cat([hr_tgt, lr_tgt], dim=0)

# --- Simplified Transformer Models (for demonstration) ---
class ToyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, use_moe=False, use_eom=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        if use_moe:
            self.ffn = EducationalMoELayerWithEOM(d_model, NUM_EXPERTS, eom_prob=0.2 if use_eom else 0.0)
        else:
            self.ffn = EducationalExpert(d_model, d_model * 4)
        self.attention = EducationalMultiHeadAttention(d_model, num_heads)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.use_moe = use_moe
        self.layer_norm = nn.LayerNorm(d_model)
        
    def forward(self, src):
        x = self.embedding(src)
        attn_out, _ = self.attention(x)
        x = self.layer_norm(x + attn_out) # Residual connection + Norm
        
        if self.use_moe:
            ffn_out, load_loss = self.ffn(x)
        else:
            ffn_out = self.ffn(x)
            load_loss = 0 # No load loss for dense model
            
        x = self.layer_norm(x + ffn_out) # Residual connection + Norm
        return self.fc_out(x), load_loss

# --- Training Loop ---
def train_model(model, name):
    print(f"--- Training {name} Model ---")
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    hr_losses, lr_losses = [], []
    steps = list(range(0, 201, 10))
    
    for step in steps:
        if step > 0:
            for _ in range(10): # Train for 10 iterations before eval
                optimizer.zero_grad()
                # Simple batching
                indices = random.sample(range(all_src.size(0)), 32)
                batch_src = all_src[indices]
                batch_tgt = all_tgt[indices]
                
                output, load_loss = model(batch_src)
                # Reshape for loss calculation
                loss = criterion(output.view(-1, VOCAB_SIZE), batch_tgt.view(-1))
                total_loss = loss + 0.01 * load_loss # Add load balancing loss
                total_loss.backward()
                optimizer.step()
        
        # Evaluate on validation sets
        with torch.no_grad():
            hr_out, _ = model(hr_val_src)
            hr_loss = criterion(hr_out.view(-1, VOCAB_SIZE), hr_val_tgt.view(-1))
            hr_losses.append(hr_loss.item())
            
            lr_out, _ = model(lr_val_src)
            lr_loss = criterion(lr_out.view(-1, VOCAB_SIZE), lr_val_tgt.view(-1))
            lr_losses.append(lr_loss.item())
            
    return steps, hr_losses, lr_losses

# Instantiate and train models
dense_model = ToyTransformer(VOCAB_SIZE, D_MODEL, NUM_HEADS, use_moe=False)
moe_model = ToyTransformer(VOCAB_SIZE, D_MODEL, NUM_HEADS, use_moe=True)
moe_eom_model = ToyTransformer(VOCAB_SIZE, D_MODEL, NUM_HEADS, use_moe=True, use_eom=True)

steps_d, hr_loss_d, lr_loss_d = train_model(dense_model, "Dense")
steps_m, hr_loss_m, lr_loss_m = train_model(moe_model, "MoE")
steps_e, hr_loss_e, lr_loss_e = train_model(moe_eom_model, "MoE with EOM")

# --- Plotting the Results ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(steps_d, lr_loss_d, 'o-', label='Dense')
ax1.plot(steps_m, lr_loss_m, 'o-', label='MoE')
ax1.plot(steps_e, lr_loss_e, 'o-', label='MoE with EOM')
ax1.set_title("Low-Resource Validation Loss ('eng' -> 'wol')")
ax1.set_xlabel("Training Steps")
ax1.set_ylabel("Cross-Entropy Loss (Perplexity Proxy)")
ax1.set_ylim(bottom=1.0, top=5.0) # Adjust ylim to see overfitting clearly
ax1.legend()
ax1.grid(True)

ax2.plot(steps_d, hr_loss_d, 'o-', label='Dense')
ax2.plot(steps_m, hr_loss_m, 'o-', label='MoE')
ax2.plot(steps_e, hr_loss_e, 'o-', label='MoE with EOM')
ax2.set_title("High-Resource Validation Loss ('eng' -> 'fra')")
ax2.set_xlabel("Training Steps")
ax2.legend()
ax2.grid(True)

plt.suptitle("Visualizing MoE Overfitting and EOM Regularization")
plt.show()

**Analysis:** The plot for the Low-Resource pair should clearly show the MoE model's validation loss starting to increase after an initial period of learning, while the Dense and EOM-regularized MoE models maintain a downward or stable trend. This demonstrates the overfitting problem described in the paper and the effectiveness of their proposed solution.

## Section 6: Research Context & Extensions

A major goal of the NLLB project is to make translation technology accessible. This involves not just training a massive model, but also making its capabilities available in a practical form. The paper discusses distilling the 54.5B parameter MoE model into smaller, faster dense models for deployment, such as for the Wikipedia Content Translation tool.

### 6.1 Model Distillation: From Giant to Practical

We'll demonstrate **Offline, Sequence-Level Knowledge Distillation**. The process is:
1.  Train a large, powerful **teacher** model.
2.  Use the teacher to translate a large monolingual corpus, creating a synthetic, "silver" parallel dataset.
3.  Train a smaller **student** model from scratch on this silver dataset.

The student learns to mimic the input-output behavior of the teacher, often achieving much better performance than a student of the same size trained only on the original, smaller "gold" dataset.

In [ ]:
# --- Distillation Demonstration Setup ---

# We'll reuse the high-resource data as our "gold" training data
gold_src, gold_tgt = hr_src, hr_tgt

# We'll create a large monolingual corpus for the teacher to translate
mono_src, _ = generate_toy_data(VOCAB_SIZE, VOCAB_SIZE, 10000, SEQ_LEN)

# 1. Train the Teacher Model (we'll use our pre-trained MoE+EOM model as the teacher)
teacher_model = moe_eom_model
teacher_model.eval() # Set to evaluation mode
print("Step 1: Teacher model is ready (using pre-trained MoE+EOM model).")

# 2. Generate the Silver Dataset
with torch.no_grad():
    silver_logits, _ = teacher_model(mono_src)
    # Get the translated sequence by taking the argmax
    silver_tgt = torch.argmax(silver_logits, dim=-1)

silver_src = mono_src
print(f"Step 2: Generated silver dataset with {silver_src.shape[0]} samples.")

# 3. Train Student Models
# Student A: Trained only on the original, smaller gold data
student_A = ToyTransformer(VOCAB_SIZE, d_model=16, num_heads=2, use_moe=False)
# Student B: Trained on the larger, silver dataset generated by the teacher
student_B = ToyTransformer(VOCAB_SIZE, d_model=16, num_heads=2, use_moe=False)

def simple_train_loop(model, src_data, tgt_data, steps=500):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    for _ in range(steps):
        optimizer.zero_grad()
        indices = random.sample(range(src_data.size(0)), 32)
        batch_src, batch_tgt = src_data[indices], tgt_data[indices]
        output, _ = model(batch_src)
        loss = criterion(output.view(-1, VOCAB_SIZE), batch_tgt.view(-1))
        loss.backward()
        optimizer.step()
    return model

print("\nStep 3: Training student models...")
student_A = simple_train_loop(student_A, gold_src, gold_tgt)
print("  - Student A (trained on gold data) is finished.")
student_B = simple_train_loop(student_B, silver_src, silver_tgt)
print("  - Student B (trained on silver data) is finished.")

# 4. Evaluate and Compare
def evaluate_accuracy(model, src_data, tgt_data):
    with torch.no_grad():
        output, _ = model(src_data)
        preds = torch.argmax(output, dim=-1)
        # Calculate token-level accuracy
        correct = (preds == tgt_data).sum().item()
        total = tgt_data.numel()
        return correct / total
        
acc_A = evaluate_accuracy(student_A, hr_val_src, hr_val_tgt)
acc_B = evaluate_accuracy(student_B, hr_val_src, hr_val_tgt)

print("\n--- Distillation Results ---")
print(f"Student A (Gold Data) Accuracy: {acc_A:.2%}")
print(f"Student B (Silver Data) Accuracy: {acc_B:.2%}")
if acc_B > acc_A:
    print("\nConclusion: The distilled student (B) outperforms the one trained on original data (A).")

### 6.2 Final Thoughts and Future Directions

The NLLB project provides a powerful blueprint for scaling multilingual technologies, but as the paper and lecture conclude, many challenges remain.

- **Beyond Text:** The future of translation is multimodal. Projects like SeamlessM4T are already extending these ideas to speech, but this introduces new challenges in data collection (for unwritten languages) and modeling.

- **True Human-Centricity:** While NLLB began with a human-centered approach, deploying these models reveals further needs. How can models be adapted to specific dialects, domains (like medical or legal text), or levels of formality? This requires ongoing collaboration with speaker communities.

- **Democratization of Scale:** Training a 54.5B parameter model is beyond the reach of most researchers. Techniques like distillation are a step forward, but further research into parameter-efficient training, modular model design, and open collaboration are needed to ensure the entire community can contribute to and benefit from these powerful multilingual systems.

# No Language Left Behind: A Deep Dive into Scaling Machine Translation

### An Interactive Educational Notebook

This notebook explores the core concepts from the research paper **"No Language Left Behind: Scaling Human-Centered Machine Translation"** by the NLLB Team at Meta AI. We will deconstruct the key innovations that enabled the creation of a machine translation model covering over 200 languages, with a special focus on low-resource languages.

## Section 1: Overview & Prerequisites

### 1.1 Summary of the Research

The "No Language Left Behind" (NLLB) project tackled the immense challenge of creating a single, high-quality machine translation (MT) system for over 200 languages, many of which are "low-resource" (meaning they lack large-scale digital text data). The work represents a significant leap from previous systems that covered around 100 languages.

The core contributions can be broken down into three main areas:

1.  **Data Curation & Creation:** Recognizing that data is the primary bottleneck, the project developed novel techniques to create vast training and evaluation datasets. This included:
    *   **FLORES-200:** A high-quality, human-translated evaluation benchmark covering all target languages, enabling reliable progress measurement.
    *   **Large-Scale Bitext Mining:** A sophisticated pipeline to find parallel sentences (translations) from massive, noisy web data (Common Crawl). This pipeline relies on highly accurate Language Identification (LID) and advanced sentence encoders (LASER3) built using a teacher-student distillation approach.
    *   **Backtranslation:** Using strong initial models to generate synthetic training data for low-resource languages, further boosting their data availability.

2.  **Modeling Innovations:** To handle the scale and diversity of 200+ languages, the project introduced advanced modeling techniques:
    *   **Mixture-of-Experts (MoE) Models:** Using a 54.5 billion parameter Sparsely Gated MoE Transformer to massively increase model capacity without a proportional increase in computational cost, allowing experts to specialize in different languages or language families.
    *   **Advanced Regularization & Curriculum Learning:** Developing novel methods like MoE Expert Output Masking (EOM) and phased training curricula to combat the severe overfitting that large models experience on low-resource languages.

3.  **Holistic & Human-Centered Evaluation:** Going beyond standard metrics like BLEU, the project focused on:
    *   **Standardized Human Evaluation:** Using the XSTS protocol with calibration sets to ensure human judgments of quality were consistent and comparable across dozens of language pairs.
    *   **Toxicity Detection:** Creating comprehensive toxicity wordlists for all 200 languages to measure and mitigate the risk of models adding offensive content during translation.
    
This notebook will walk through the mathematical foundations, prerequisite algorithms, and core implementations of these key contributions.

### 1.2 Prerequisites

To fully grasp the concepts in this notebook, the following background knowledge is recommended.

**Mathematical Concepts:**
- **Linear Algebra:** Vector spaces, dot products, matrix multiplication, norms.
- **Calculus:** Partial derivatives, gradients, the chain rule (for backpropagation).
- **Probability & Statistics:** Probability distributions, conditional probability, expectation.
- **Information Theory:** Entropy, cross-entropy, perplexity.

**Machine Learning & Computer Science Concepts:**
- **Neural Networks:** Basic architecture (layers, neurons, activation functions), loss functions, gradient descent.
- **Natural Language Processing (NLP):** Tokenization (subword units), embeddings, sequence-to-sequence models.
- **The Transformer Architecture:** Self-attention mechanism, multi-head attention, positional encodings, encoder-decoder structure.
- **Knowledge Distillation:** The concept of a larger "teacher" model training a smaller "student" model.
- **Mixture of Experts (MoE):** Basic understanding of conditional computation where only parts of a network are activated per input.

### 1.3 Notebook Structure & Learning Objectives

**Hierarchy of Topics:**
1.  **Overview & Prerequisites**: Setting the stage for our deep dive.
2.  **Mathematical Foundations**: Implementing key math concepts like Cosine Similarity and Cross-Entropy from scratch.
3.  **Prerequisite Algorithms**: Building a foundational Transformer model from scratch to understand the base architecture.
4.  **Core Research Content**: Implementing the key NLLB innovations:
    - The Bitext Mining pipeline philosophy (LID, Sentence Encoders).
    - Sparsely Gated Mixture-of-Experts (MoE) layers.
    - Advanced regularization with Expert Output Masking (EOM).
5.  **Experimental Analysis**: Visually reproducing key results from the paper, such as the overfitting problem in MoE models and the effect of different data sources.
6.  **Research Context & Extensions**: Exploring model distillation for creating smaller, practical models and discussing the challenges of dialectal translation.

**Learning Objectives:**
- **Understand** the core challenges of low-resource machine translation.
- **Implement** a Transformer model and a Mixture-of-Experts layer.
- **Grasp** the importance of data mining and augmentation for large-scale multilingual models.
- **Analyze** the trade-offs between model scale, performance, and overfitting.
- **Appreciate** the need for human-centered and safety-focused evaluation.

**Estimated Time:** 2-3 hours.

## Section 2: Mathematical Foundations

Before diving into complex models, let's implement and visualize the core mathematical tools used in the NLLB paper, particularly for the bitext mining and model training processes.

### 2.1 Cosine Similarity: Measuring Meaning

The entire bitext mining pipeline hinges on finding sentences with similar meanings across languages. This is achieved by representing sentences as high-dimensional vectors (embeddings) and measuring their similarity. The most common metric for this is **Cosine Similarity**, which measures the cosine of the angle between two vectors. A value of 1 means the vectors point in the same direction (identical meaning), 0 means they are orthogonal (unrelated), and -1 means they are opposite.

The formula is: 
$$ \text{similarity} = \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|} = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \sqrt{\sum_{i=1}^{n} B_i^2}} $$

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def educational_cosine_similarity(vec_a, vec_b):
    """
    Clear, step-by-step implementation of cosine similarity for understanding.
    - Based directly on the mathematical formula.
    - Uses basic loops and operations.
    """
    dot_product = 0.0
    norm_a = 0.0
    norm_b = 0.0
    
    for i in range(len(vec_a)):
        dot_product += vec_a[i] * vec_b[i]
        norm_a += vec_a[i]**2
        norm_b += vec_b[i]**2
        
    norm_a = np.sqrt(norm_a)
    norm_b = np.sqrt(norm_b)
    
    if norm_a == 0 or norm_b == 0:
        return 0.0 # Avoid division by zero
        
    return dot_product / (norm_a * norm_b)

def optimized_cosine_similarity(vec_a, vec_b):
    """
    Efficient, vectorized implementation using NumPy.
    """
    # Ensure inputs are NumPy arrays
    vec_a = np.asarray(vec_a)
    vec_b = np.asarray(vec_b)
    
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    
    if norm_a == 0 or norm_b == 0:
        return 0.0
        
    return dot_product / (norm_a * norm_b)

# --- Interactive Visualization ---
@widgets.interact(angle=widgets.FloatSlider(value=45, min=0, max=180, step=1, description='Angle (θ):'))
def interactive_cosine_explorer(angle):
    """
    Interactive widget to visualize cosine similarity between two 2D vectors.
    """
    # Vector A is fixed along the x-axis
    vec_a = np.array([1, 0])
    
    # Vector B is rotated based on the angle slider
    angle_rad = np.deg2rad(angle)
    vec_b = np.array([np.cos(angle_rad), np.sin(angle_rad)])
    
    # Calculate similarity
    sim_educational = educational_cosine_similarity(vec_a, vec_b)
    sim_optimized = optimized_cosine_similarity(vec_a, vec_b)
    
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.arrow(0, 0, vec_a[0], vec_a[1], head_width=0.05, head_length=0.1, fc='blue', ec='blue', label='Vector A')
    ax.arrow(0, 0, vec_b[0], vec_b[1], head_width=0.05, head_length=0.1, fc='red', ec='red', label='Vector B')
    
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-0.2, 1.2)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True)
    ax.axhline(0, color='black',linewidth=0.5)
    ax.axvline(0, color='black',linewidth=0.5)
    ax.set_title(f'Cosine Similarity: {sim_optimized:.2f}')
    ax.legend()
    plt.show()

    print(f"Educational implementation result: {sim_educational:.4f}")
    print(f"Optimized implementation result:   {sim_optimized:.4f}")

### 2.2 Cross-Entropy Loss: Guiding the Model's Learning

The machine translation model learns by trying to predict the next word in the translated sentence. The **Cross-Entropy Loss** function measures how well the probability distribution predicted by the model matches the actual target distribution (which is a one-hot vector, where the correct word has a probability of 1 and all others have 0).

For a single prediction, the formula is:
$$ H(p, q) = - \sum_{i=1}^{C} p(x_i) \log q(x_i) $$ 

Where:
- $ C $ is the number of classes (the vocabulary size).
- $ p(x_i) $ is the true probability of class $i$ (1 for the correct word, 0 for all others).
- $ q(x_i) $ is the model's predicted probability for class $i$ (output of a Softmax function).

This simplifies to just the negative log probability of the correct word: $ -\log(q(x_{\text{correct}})) $. Minimizing this loss forces the model to assign a higher probability to the correct word.

In [ ]:
def educational_softmax(logits):
    """Clear implementation of the softmax function."""
    exps = [np.exp(logit) for logit in logits]
    sum_of_exps = sum(exps)
    softmax_probs = [exp / sum_of_exps for exp in exps]
    return softmax_probs

def educational_cross_entropy_loss(predicted_probs, target_index):
    """Clear implementation of cross-entropy loss for a single example."""
    # The loss is simply the negative log of the probability of the correct class
    correct_prob = predicted_probs[target_index]
    return -np.log(correct_prob)

def optimized_cross_entropy_loss(logits_tensor, target_index_tensor):
    """Efficient implementation using PyTorch's built-in functions."""
    # PyTorch's F.cross_entropy handily combines log_softmax and NLLLoss
    return F.cross_entropy(logits_tensor, target_index_tensor)

# --- Example Usage and Verification ---
# Model's raw output for a vocabulary of 5 words
logits = [2.0, 1.0, 0.1, -1.0, 0.5]
target_index = 0 # The correct word is the first one

# Educational Path
probs_edu = educational_softmax(logits)
loss_edu = educational_cross_entropy_loss(probs_edu, target_index)

# Optimized Path (using PyTorch)
logits_pt = torch.tensor([logits], dtype=torch.float32) # Add batch dimension
target_pt = torch.tensor([target_index], dtype=torch.long)
loss_pt = optimized_cross_entropy_loss(logits_pt, target_pt).item()

print(f"Logits: {logits}")
print(f"Target Index: {target_index}\n")
print(f"Probabilities (Educational): {[f'{p:.3f}' for p in probs_edu]}\n")
print(f"Educational Cross-Entropy Loss: {loss_edu:.4f}")
print(f"Optimized Cross-Entropy Loss:   {loss_pt:.4f}")

## Section 3: Prerequisite Algorithms

The NLLB-200 model is a variant of the Transformer. To understand its architecture and the innovations applied to it, we must first understand the original "Attention Is All You Need" Transformer model.

### 3.1 The Transformer Architecture (From Scratch)

The Transformer, introduced by Vaswani et al. (2017), revolutionized sequence-to-sequence tasks. Its core innovation is the **self-attention mechanism**, which allows the model to weigh the importance of different words in the input sequence when processing a specific word. This replaces the recurrent connections of models like LSTMs, enabling massive parallelization.

We will implement the key components from scratch to build a functional (though not optimized) Transformer block.

**Key Components:**
1.  **Scaled Dot-Product Attention:** The core of the attention mechanism. It computes attention scores for a set of queries ($Q$), keys ($K$), and values ($V$).
    $$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$ 
2.  **Multi-Head Attention:** Runs the attention mechanism in parallel multiple times (with different linear projections) and concatenates the results, allowing the model to focus on different aspects of the sequence.
3.  **Position-wise Feed-Forward Network:** A simple two-layer fully connected network applied independently to each position.
4.  **Positional Encoding:** Since the model contains no recurrence, we add information about the position of tokens in the sequence to the input embeddings.

In [ ]:
import torch
import torch.nn as nn
import math

class EducationalMultiHeadAttention(nn.Module):
    """
    Educational implementation of Multi-Head Attention.
    Focuses on clarity over performance.
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear layers for Query, Key, Value, and the final output
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """The core attention mechanism."""
        # 1. MatMul Q and K.T
        attn_scores = torch.matmul(Q, K.transpose(-2, -1))
        
        # 2. Scale by sqrt(d_k)
        attn_scores = attn_scores / math.sqrt(self.d_k)
        
        # 3. Apply mask (optional, for decoder)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        # 4. Apply softmax
        attn_probs = torch.softmax(attn_scores, dim=-1)
        
        # 5. MatMul with V
        output = torch.matmul(attn_probs, V)
        return output, attn_probs

    def forward(self, x):
        batch_size, seq_length, _ = x.size()
        
        # 1. Pass input through linear layers
        Q = self.w_q(x)
        K = self.w_k(x)
        V = self.w_v(x)
        
        # 2. Reshape for multi-head attention
        # (batch_size, seq_length, d_model) -> (batch_size, num_heads, seq_length, d_k)
        Q = Q.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. Apply scaled dot-product attention
        attn_output, attn_probs = self.scaled_dot_product_attention(Q, K, V)
        
        # 4. Concatenate heads and pass through final linear layer
        # (batch_size, num_heads, seq_length, d_k) -> (batch_size, seq_length, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        output = self.w_o(attn_output)
        
        return output, attn_probs

# --- Let's test it ---
d_model = 512
num_heads = 8
batch_size = 2
seq_len = 10

attention = EducationalMultiHeadAttention(d_model, num_heads)
input_tensor = torch.randn(batch_size, seq_len, d_model)
output, attn_probs = attention(input_tensor)

print(f"Input shape: {input_tensor.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention probabilities shape (one head): {attn_probs.shape}")

# Visualize attention for the first head of the first batch item
plt.figure(figsize=(8, 6))
plt.imshow(attn_probs[0, 0].detach().numpy(), cmap='viridis')
plt.xlabel("Key Position")
plt.ylabel("Query Position")
plt.title("Self-Attention Weights (Head 1, Batch 1)")
plt.colorbar()
plt.show()

## Section 4: Core Research Content

Now we will implement simplified, educational versions of the core modeling innovations from the NLLB paper: Sparsely Gated Mixture-of-Experts, the EOM regularization technique, and the concept of curriculum learning.

### 4.1 Sparsely-Gated Mixture-of-Experts (MoE) Layer

A standard Transformer uses the same Feed-Forward Network (FFN) for every token. An MoE model replaces this FFN with a set of parallel FFNs, called "experts". For each token, a small trainable "gating network" decides which expert(s) to send it to. The NLLB model uses a **Top-2 Gating** strategy, where each token is processed by the two experts that the gating network scores most highly.

The output is a weighted sum of the outputs from the selected experts:
$$ \text{MoE}(x_t) = \sum_{e=1}^{E} G(x_t)_e \cdot \text{FFN}_e(x_t) $$

Where $G(x_t)_e$ is the gate's weight for expert $e$. In Top-2 gating, only the top two weights are non-zero.

A crucial component is the **Load Balancing Loss**, an auxiliary loss that encourages the gating network to send a roughly equal number of tokens to each expert, preventing a scenario where only a few experts are ever used.

In [ ]:
class EducationalExpert(nn.Module):
    """A simple Feed-Forward Network, representing one expert."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.activation = nn.ReLU()
        
    def forward(self, x):
        return self.fc2(self.activation(self.fc1(x)))

class EducationalMoELayer(nn.Module):
    """
    Educational implementation of a Top-2 Sparsely Gated MoE layer.
    """
    def __init__(self, d_model, num_experts, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        
        # The list of expert networks
        self.experts = nn.ModuleList([EducationalExpert(d_model, d_model * 4) for _ in range(num_experts)])
        
        # The gating network is a simple linear layer
        self.gate = nn.Linear(d_model, num_experts)
        
    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        batch_size, seq_len, d_model = x.shape
        x_flat = x.view(-1, d_model) # Reshape to (num_tokens, d_model)
        num_tokens = x_flat.shape[0]

        # 1. Get gating scores for each token
        gate_logits = self.gate(x_flat) # (num_tokens, num_experts)
        gate_probs = F.softmax(gate_logits, dim=1)
        
        # 2. Select Top-K experts for each token
        top_k_probs, top_k_indices = torch.topk(gate_probs, self.top_k, dim=1)
        
        # 3. Normalize the weights of the top-k experts
        top_k_probs = top_k_probs / torch.sum(top_k_probs, dim=1, keepdim=True)
        
        # 4. Route tokens to experts and combine outputs
        final_output = torch.zeros_like(x_flat)
        for i in range(num_tokens):
            token_input = x_flat[i]
            token_output = torch.zeros(d_model)
            for k in range(self.top_k):
                expert_idx = top_k_indices[i, k].item()
                expert_weight = top_k_probs[i, k]
                
                # Get output from the selected expert
                expert_output = self.experts[expert_idx](token_input)
                
                # Weight the expert's output
                token_output += expert_weight * expert_output
            final_output[i] = token_output
            
        # --- Load Balancing Loss Calculation (simplified) ---
        # This encourages gates to use all experts equally.
        # The paper uses a more complex formula, but this captures the idea.
        tokens_per_expert = F.one_hot(top_k_indices[:, 0], num_classes=self.num_experts).float().sum(0)
        load_balancing_loss = torch.std(tokens_per_expert) / torch.mean(tokens_per_expert)
        
        return final_output.view(batch_size, seq_len, d_model), load_balancing_loss, gate_probs

# --- Let's test and visualize the gating ---
d_model = 64
num_experts = 8
moe_layer = EducationalMoELayer(d_model, num_experts)
input_tensor = torch.randn(1, 10, d_model) # 1 sentence of 10 tokens
output, loss, gate_probs = moe_layer(input_tensor)

print(f"Input shape: {input_tensor.shape}")
print(f"Output shape: {output.shape}")
print(f"Load Balancing Loss: {loss.item():.4f}\n")

plt.figure(figsize=(10, 4))
plt.imshow(gate_probs.detach().numpy().T, cmap='viridis', aspect='auto')
plt.xlabel("Token Position in Sequence")
plt.ylabel("Expert ID")
plt.title("Gating Probabilities per Token")
plt.colorbar(label="Probability")
plt.show()

### 4.2 Expert Output Masking (EOM) for Regularization

As the paper notes, large MoE models are prone to overfitting on low-resource languages. Standard dropout is not always sufficient. The paper proposes **MoE Expert Output Masking (EOM)**, a specialized regularization technique.

The idea is simple: for a random fraction of tokens, the outputs of the selected experts are masked (zeroed out) *before* they are combined. This forces the model to rely more on its residual connection (the skip-connection around the MoE block) and reduces co-adaptation between experts.

Let's modify our MoE layer to include this technique.

In [ ]:
class EducationalMoELayerWithEOM(EducationalMoELayer):
    """
    Extends the MoE layer to include Expert Output Masking (EOM).
    """
    def __init__(self, d_model, num_experts, top_k=2, eom_prob=0.1):
        super().__init__(d_model, num_experts, top_k)
        self.eom_prob = eom_prob

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape
        x_flat = x.view(-1, d_model)
        num_tokens = x_flat.shape[0]

        gate_logits = self.gate(x_flat)
        gate_probs = F.softmax(gate_logits, dim=1)
        top_k_probs, top_k_indices = torch.topk(gate_probs, self.top_k, dim=1)
        top_k_probs = top_k_probs / torch.sum(top_k_probs, dim=1, keepdim=True)

        final_output = torch.zeros_like(x_flat)
        for i in range(num_tokens):
            token_input = x_flat[i]
            token_output = torch.zeros(d_model)
            
            # --- EOM Logic ---
            # Decide if we should mask the expert outputs for this token
            apply_eom = torch.rand(1) < self.eom_prob
            
            for k in range(self.top_k):
                expert_idx = top_k_indices[i, k].item()
                expert_weight = top_k_probs[i, k]
                expert_output = self.experts[expert_idx](token_input)
                
                if not apply_eom:
                    token_output += expert_weight * expert_output
                # If apply_eom is True, we add nothing, effectively masking the output.
                
            final_output[i] = token_output
        
        # Load balancing loss is calculated the same way, as it depends on gate scores, not outputs
        tokens_per_expert = F.one_hot(top_k_indices[:, 0], num_classes=self.num_experts).float().sum(0)
        load_balancing_loss = torch.std(tokens_per_expert) / torch.mean(tokens_per_expert)

        return final_output.view(batch_size, seq_len, d_model), load_balancing_loss

# --- Demonstration ---
eom_prob = 0.5 # Set high for clear demonstration
moe_eom_layer = EducationalMoELayerWithEOM(d_model, num_experts, eom_prob=eom_prob)
output_eom, _ = moe_eom_layer(input_tensor)

print("EOM forces some token outputs to be zero (relying on the residual connection).")
print("Original Model Output (first 5 tokens):\n", output[0, :5, :4].detach().numpy().round(2))
print("\nModel with EOM Output (first 5 tokens):\n", output_eom[0, :5, :4].detach().numpy().round(2))

## Section 5: Experimental Analysis

A key finding of the paper is that while MoE models offer massive capacity, they overfit easily on low-resource language pairs. Let's create a simplified experiment to visualize this phenomenon, reproducing the core insight of Figure 17 from the paper.

### 5.1 Visualizing Overfitting in Low-Resource MoE Training

We'll set up a toy translation task with two language pairs:
1.  **High-Resource (`eng` -> `fra`):** A larger dataset.
2.  **Low-Resource (`eng` -> `wol`):** A much smaller dataset.

We will train three small models on this combined data:
1.  A **Dense** Transformer.
2.  An **MoE** Transformer.
3.  An **MoE** Transformer with **EOM** regularization.

We will then plot the validation loss (a proxy for perplexity) for both the high-resource and low-resource pairs over the course of training. We expect to see the MoE model's loss on the low-resource pair start to increase after an initial decrease, indicating overfitting. The EOM and Dense models should show more stable performance.

In [ ]:
import random

# --- Toy Data Generation ---
def generate_toy_data(src_vocab_size, tgt_vocab_size, num_samples, seq_len):
    # Simple task: learn to reverse a sequence
    src_data = torch.randint(1, src_vocab_size, (num_samples, seq_len))
    tgt_data = torch.flip(src_data, [1])
    return src_data, tgt_data

VOCAB_SIZE = 100
SEQ_LEN = 8
D_MODEL = 32
NUM_HEADS = 4
NUM_EXPERTS = 4

# High-resource: 2000 samples
hr_src, hr_tgt = generate_toy_data(VOCAB_SIZE, VOCAB_SIZE, 2000, SEQ_LEN)
# Low-resource: 200 samples
lr_src, lr_tgt = generate_toy_data(VOCAB_SIZE, VOCAB_SIZE, 200, SEQ_LEN)

# Create validation sets
hr_val_src, hr_val_tgt = generate_toy_data(VOCAB_SIZE, VOCAB_SIZE, 100, SEQ_LEN)
lr_val_src, lr_val_tgt = generate_toy_data(VOCAB_SIZE, VOCAB_SIZE, 100, SEQ_LEN)

# Combine data for training
all_src = torch.cat([hr_src, lr_src], dim=0)
all_tgt = torch.cat([hr_tgt, lr_tgt], dim=0)

# --- Simplified Transformer Models (for demonstration) ---
class ToyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, use_moe=False, use_eom=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        if use_moe:
            self.ffn = EducationalMoELayerWithEOM(d_model, NUM_EXPERTS, eom_prob=0.2 if use_eom else 0.0)
        else:
            self.ffn = EducationalExpert(d_model, d_model * 4)
        self.attention = EducationalMultiHeadAttention(d_model, num_heads)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.use_moe = use_moe
        self.layer_norm = nn.LayerNorm(d_model)
        
    def forward(self, src):
        x = self.embedding(src)
        attn_out, _ = self.attention(x)
        x = self.layer_norm(x + attn_out) # Residual connection + Norm
        
        if self.use_moe:
            ffn_out, load_loss = self.ffn(x)
        else:
            ffn_out = self.ffn(x)
            load_loss = 0 # No load loss for dense model
            
        x = self.layer_norm(x + ffn_out) # Residual connection + Norm
        return self.fc_out(x), load_loss

# --- Training Loop ---
def train_model(model, name):
    print(f"--- Training {name} Model ---")
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    hr_losses, lr_losses = [], []
    steps = list(range(0, 201, 10))
    
    for step in steps:
        if step > 0:
            for _ in range(10): # Train for 10 iterations before eval
                optimizer.zero_grad()
                # Simple batching
                indices = random.sample(range(all_src.size(0)), 32)
                batch_src = all_src[indices]
                batch_tgt = all_tgt[indices]
                
                output, load_loss = model(batch_src)
                # Reshape for loss calculation
                loss = criterion(output.view(-1, VOCAB_SIZE), batch_tgt.view(-1))
                total_loss = loss + 0.01 * load_loss # Add load balancing loss
                total_loss.backward()
                optimizer.step()
        
        # Evaluate on validation sets
        with torch.no_grad():
            hr_out, _ = model(hr_val_src)
            hr_loss = criterion(hr_out.view(-1, VOCAB_SIZE), hr_val_tgt.view(-1))
            hr_losses.append(hr_loss.item())
            
            lr_out, _ = model(lr_val_src)
            lr_loss = criterion(lr_out.view(-1, VOCAB_SIZE), lr_val_tgt.view(-1))
            lr_losses.append(lr_loss.item())
            
    return steps, hr_losses, lr_losses

# Instantiate and train models
dense_model = ToyTransformer(VOCAB_SIZE, D_MODEL, NUM_HEADS, use_moe=False)
moe_model = ToyTransformer(VOCAB_SIZE, D_MODEL, NUM_HEADS, use_moe=True)
moe_eom_model = ToyTransformer(VOCAB_SIZE, D_MODEL, NUM_HEADS, use_moe=True, use_eom=True)

steps_d, hr_loss_d, lr_loss_d = train_model(dense_model, "Dense")
steps_m, hr_loss_m, lr_loss_m = train_model(moe_model, "MoE")
steps_e, hr_loss_e, lr_loss_e = train_model(moe_eom_model, "MoE with EOM")

# --- Plotting the Results ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(steps_d, lr_loss_d, 'o-', label='Dense')
ax1.plot(steps_m, lr_loss_m, 'o-', label='MoE')
ax1.plot(steps_e, lr_loss_e, 'o-', label='MoE with EOM')
ax1.set_title("Low-Resource Validation Loss ('eng' -> 'wol')")
ax1.set_xlabel("Training Steps")
ax1.set_ylabel("Cross-Entropy Loss (Perplexity Proxy)")
ax1.set_ylim(bottom=1.0, top=5.0) # Adjust ylim to see overfitting clearly
ax1.legend()
ax1.grid(True)

ax2.plot(steps_d, hr_loss_d, 'o-', label='Dense')
ax2.plot(steps_m, hr_loss_m, 'o-', label='MoE')
ax2.plot(steps_e, hr_loss_e, 'o-', label='MoE with EOM')
ax2.set_title("High-Resource Validation Loss ('eng' -> 'fra')")
ax2.set_xlabel("Training Steps")
ax2.legend()
ax2.grid(True)

plt.suptitle("Visualizing MoE Overfitting and EOM Regularization")
plt.show()

**Analysis:** The plot for the Low-Resource pair should clearly show the MoE model's validation loss starting to increase after an initial period of learning, while the Dense and EOM-regularized MoE models maintain a downward or stable trend. This demonstrates the overfitting problem described in the paper and the effectiveness of their proposed solution.

## Section 6: Research Context & Extensions

A major goal of the NLLB project is to make translation technology accessible. This involves not just training a massive model, but also making its capabilities available in a practical form. The paper discusses distilling the 54.5B parameter MoE model into smaller, faster dense models for deployment, such as for the Wikipedia Content Translation tool.

### 6.1 Model Distillation: From Giant to Practical

We'll demonstrate **Offline, Sequence-Level Knowledge Distillation**. The process is:
1.  Train a large, powerful **teacher** model.
2.  Use the teacher to translate a large monolingual corpus, creating a synthetic, "silver" parallel dataset.
3.  Train a smaller **student** model from scratch on this silver dataset.

The student learns to mimic the input-output behavior of the teacher, often achieving much better performance than a student of the same size trained only on the original, smaller "gold" dataset.

In [ ]:
# --- Distillation Demonstration Setup ---

# We'll reuse the high-resource data as our "gold" training data
gold_src, gold_tgt = hr_src, hr_tgt

# We'll create a large monolingual corpus for the teacher to translate
mono_src, _ = generate_toy_data(VOCAB_SIZE, VOCAB_SIZE, 10000, SEQ_LEN)

# 1. Train the Teacher Model (we'll use our pre-trained MoE+EOM model as the teacher)
teacher_model = moe_eom_model
teacher_model.eval() # Set to evaluation mode
print("Step 1: Teacher model is ready (using pre-trained MoE+EOM model).")

# 2. Generate the Silver Dataset
with torch.no_grad():
    silver_logits, _ = teacher_model(mono_src)
    # Get the translated sequence by taking the argmax
    silver_tgt = torch.argmax(silver_logits, dim=-1)

silver_src = mono_src
print(f"Step 2: Generated silver dataset with {silver_src.shape[0]} samples.")

# 3. Train Student Models
# Student A: Trained only on the original, smaller gold data
student_A = ToyTransformer(VOCAB_SIZE, d_model=16, num_heads=2, use_moe=False)
# Student B: Trained on the larger, silver dataset generated by the teacher
student_B = ToyTransformer(VOCAB_SIZE, d_model=16, num_heads=2, use_moe=False)

def simple_train_loop(model, src_data, tgt_data, steps=500):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    for _ in range(steps):
        optimizer.zero_grad()
        indices = random.sample(range(src_data.size(0)), 32)
        batch_src, batch_tgt = src_data[indices], tgt_data[indices]
        output, _ = model(batch_src)
        loss = criterion(output.view(-1, VOCAB_SIZE), batch_tgt.view(-1))
        loss.backward()
        optimizer.step()
    return model

print("\nStep 3: Training student models...")
student_A = simple_train_loop(student_A, gold_src, gold_tgt)
print("  - Student A (trained on gold data) is finished.")
student_B = simple_train_loop(student_B, silver_src, silver_tgt)
print("  - Student B (trained on silver data) is finished.")

# 4. Evaluate and Compare
def evaluate_accuracy(model, src_data, tgt_data):
    with torch.no_grad():
        output, _ = model(src_data)
        preds = torch.argmax(output, dim=-1)
        # Calculate token-level accuracy
        correct = (preds == tgt_data).sum().item()
        total = tgt_data.numel()
        return correct / total
        
acc_A = evaluate_accuracy(student_A, hr_val_src, hr_val_tgt)
acc_B = evaluate_accuracy(student_B, hr_val_src, hr_val_tgt)

print("\n--- Distillation Results ---")
print(f"Student A (Gold Data) Accuracy: {acc_A:.2%}")
print(f"Student B (Silver Data) Accuracy: {acc_B:.2%}")
if acc_B > acc_A:
    print("\nConclusion: The distilled student (B) outperforms the one trained on original data (A).")

### 6.2 Final Thoughts and Future Directions

The NLLB project provides a powerful blueprint for scaling multilingual technologies, but as the paper and lecture conclude, many challenges remain.

- **Beyond Text:** The future of translation is multimodal. Projects like SeamlessM4T are already extending these ideas to speech, but this introduces new challenges in data collection (for unwritten languages) and modeling.

- **True Human-Centricity:** While NLLB began with a human-centered approach, deploying these models reveals further needs. How can models be adapted to specific dialects, domains (like medical or legal text), or levels of formality? This requires ongoing collaboration with speaker communities.

- **Democratization of Scale:** Training a 54.5B parameter model is beyond the reach of most researchers. Techniques like distillation are a step forward, but further research into parameter-efficient training, modular model design, and open collaboration are needed to ensure the entire community can contribute to and benefit from these powerful multilingual systems.